In [1]:
import numpy as np
import pandas as pd

np.random.seed(11)

n = 288
times = pd.date_range("2026-10-13 00:00", periods=n, freq="5min")
hours = times.hour + times.minute / 60

truth_temp = (
    20 + 2.5 * np.sin((hours - 9) / 24 * 2 * np.pi)
    + np.random.normal(0, 0.1, n)
)

truth = pd.DataFrame({
    "timestamp": times,
    "temp_c": truth_temp
})

sensor = truth.copy()
drop = np.random.choice(sensor.index, size=34, replace=False)
sensor = sensor.drop(index=drop).reset_index(drop=True)

stuck_start = 150
sensor.loc[stuck_start:stuck_start + 13, "temp_c"] = sensor.loc[stuck_start, "temp_c"]

safe = [i for i in sensor.index if not (stuck_start <= i <= stuck_start + 13)]
g = np.random.choice(safe, size=3, replace=False)
sensor.loc[g, "temp_c"] = [-40.0, 85.0, 0.0]

# Inspect the data
impossible = ~sensor["temp_c"].between(5, 40)
missing = n - len(sensor)

print("Expected readings:", n)
print("Readings received:", len(sensor))
print("Missing readings:", missing)
print("Impossible readings:", int(impossible.sum()))

display(sensor.head())

Expected readings: 288
Readings received: 254
Missing readings: 34
Impossible readings: 3


,timestamp,temp_c
0,2026-10-13 00:00:00,18.407179
1,2026-10-13 00:05:00,18.165483
2,2026-10-13 00:10:00,18.108350
3,2026-10-13 00:15:00,17.855069
4,2026-10-13 00:20:00,18.084060


In [2]:
# TASK A: EDGE CLEANING

edge = sensor.copy()

# Step 1: Remove impossible temperatures
edge = edge[edge["temp_c"].between(5, 40)].copy()

# Step 2: Detect stuck sensor readings
rolling_std = edge["temp_c"].rolling(6).std()
stuck = rolling_std.eq(0)

edge = edge.loc[~stuck].copy()

print("Original sensor readings:", len(sensor))
print("Readings after edge cleaning:", len(edge))
print("Readings removed at edge:", len(sensor) - len(edge))

display(edge.head())

Original sensor readings: 254
Readings after edge cleaning: 242
Readings removed at edge: 12


,timestamp,temp_c
0,2026-10-13 00:00:00,18.407179
1,2026-10-13 00:05:00,18.165483
2,2026-10-13 00:10:00,18.108350
3,2026-10-13 00:15:00,17.855069
4,2026-10-13 00:20:00,18.084060


In [4]:
# TASK A: CLOUD CLEANING

cloud = edge.copy().sort_values("timestamp").copy()

# Restore the original 5-minute timeline
cloud = cloud.set_index("timestamp").reindex(truth["timestamp"])

# Fill missing temperature readings using interpolation
cloud["temp_c"] = cloud["temp_c"].interpolate(method="time")

# Fill any remaining gaps at the beginning or end
cloud["temp_c"] = cloud["temp_c"].ffill().bfill()

cloud = cloud.reset_index(names="timestamp")

print("Readings after edge cleaning:", len(edge))
print("Readings after cloud cleaning:", len(cloud))
print("Missing temperatures after cloud cleaning:", cloud["temp_c"].isna().sum())

display(cloud.head())

Readings after edge cleaning: 242
Readings after cloud cleaning: 288
Missing temperatures after cloud cleaning: 0


,timestamp,temp_c
0,2026-10-13 00:00:00,18.407179
1,2026-10-13 00:05:00,18.165483
2,2026-10-13 00:10:00,18.108350
3,2026-10-13 00:15:00,17.855069
4,2026-10-13 00:20:00,18.084060


In [8]:
# TASK B: Compare MAE before and after cleaning

raw_eval = sensor.merge(
    truth, on="timestamp", suffixes=("_sensor", "_true")
)

edge_eval = edge.merge(
    truth, on="timestamp", suffixes=("_sensor", "_true")
)

cloud_eval = cloud.merge(
    truth, on="timestamp", suffixes=("_sensor", "_true")
)

raw_mae = (
    raw_eval["temp_c_sensor"] - raw_eval["temp_c_true"]
).abs().mean()

edge_mae = (
    edge_eval["temp_c_sensor"] - edge_eval["temp_c_true"]
).abs().mean()

cloud_mae = (
    cloud_eval["temp_c_sensor"] - cloud_eval["temp_c_true"]
).abs().mean()

results = pd.DataFrame({
    "Stage": [
        "Raw sensor",
        "After edge cleaning",
        "After cloud cleaning"
    ],
    "Readings evaluated": [
        len(raw_eval),
        len(edge_eval),
        len(cloud_eval)
    ],
    "MAE (°C)": [
        raw_mae,
        edge_mae,
        cloud_mae
    ]
})

display(results.round(3))

,Stage,Readings evaluated,MAE (°C)
0,Raw sensor,254,0.582
1,After edge cleaning,242,0.001
2,After cloud cleaning,288,0.014


### Task C: Edge or Cloud? — Cleaning Decisions

| Cleaning step | Edge or cloud? | Reason (real constraint) |
|---|---|---|
| Range check: detect impossible temperature values | Edge | A range check requires little memory or processing power, so the sensor can reject impossible readings immediately without spending network bandwidth sending them to the cloud. |
| Stuck sensor detection: identify repeated identical readings | Edge | The device can compare each reading with a small buffer of recent values. This uses limited memory and avoids the network cost of sending obviously faulty readings. |
| Glitch detection: identify sudden, unrealistic temperature changes | Edge | A simple threshold check needs little battery power and processing time, allowing suspicious readings to be flagged before transmission and reducing unnecessary network traffic. |
| Gap filling: fill missing timestamps or readings | Cloud | Gap filling may need readings before and after a missing timestamp, so cloud processing can use a longer history and more memory than a small sensor device. |

In [10]:

# Task D: Feature Engineering

# Start with the true temperature timeline
features = truth.copy()

# Feature 1: Hour of the day
features["hour"] = (
    features["timestamp"].dt.hour
    + features["timestamp"].dt.minute / 60
)

# Feature 2: One-hour rolling average (12 readings of 5 minutes each)
features["rolling_avg_1h"] = (
    features["temp_c"].rolling(window=12, min_periods=12).mean()
)

# Feature 3: Change since the previous reading
features["temp_change"] = features["temp_c"].diff()

# Target: will the room be warmer than 21°C in one hour?
features["target"] = (
    features["temp_c"].shift(-12) > 21
).astype(float)

# The final 12 readings have no future target, so remove them.
features = features.iloc[:-12].copy()

# Remove rows with missing feature values
features = features.dropna().reset_index(drop=True)

# Check correlations with the target
correlations = features[
    ["hour", "rolling_avg_1h", "temp_change", "target"]
].corr(numeric_only=True)["target"].drop("target")

print("Feature correlations with the one-hour-ahead target:")
display(correlations.to_frame(name="Correlation with target"))

print("Number of usable rows:", len(features))


Feature correlations with the one-hour-ahead target:


,Correlation with target
hour,0.264944
rolling_avg_1h,0.779360
temp_change,0.040123


Number of usable rows: 265


### Task E: Privacy — Protecting Room Occupancy Information

Temperature changes can indirectly reveal when people enter or leave a room. For example, a sudden temperature increase may occur when people enter, while a change in temperature patterns may suggest that a room is empty.

Before sending data to the cloud, I would aggregate temperature readings into 15-minute averages at the edge device instead of transmitting every five-minute reading. I would also remove exact timestamps and send only the time interval and average temperature. If occupancy information is not needed for temperature prediction, I would avoid transmitting features that could reveal individual entry and exit patterns.

This approach reduces the amount of detailed information leaving the building and makes it harder to infer precise occupancy events. It can also reduce network traffic. However, aggregation loses short-term temperature changes, and removing exact timestamps reduces the ability to identify sudden events or make precise predictions. Therefore, I would keep enough time information to support temperature control while avoiding unnecessary detailed records.

This is a balance between privacy, prediction accuracy and efficient data transmission.